In [3]:
# Importa a biblioteca específica do Colab para acessar o Google Drive
from google.colab import drive
import pandas as pd
import numpy as np
import glob
import os
import time
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from tensorflow import keras

from sklearn.metrics import classification_report, confusion_matrix

from keras.models import Sequential
from keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Solicitará permissão para acessar o seu Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Caminho da pasta no seu Google Drive (note que o Colab mapeia "Meu Drive" como "MyDrive")
caminho_pasta = '/content/drive/MyDrive/Fatec/TCC/dataset'

# Mapeia todos os arquivos que terminam com .csv dentro da pasta
arquivos_csv = glob.glob(os.path.join(caminho_pasta, "*.csv"))

print(f"Encontrados {len(arquivos_csv)} arquivos para importação.")

# Lê cada um dos 8 arquivos e os junta em um único DataFrame
lista_dfs = []
for arquivo in arquivos_csv:
    df_temp = pd.read_csv(arquivo)
    lista_dfs.append(df_temp)

# Concatena todos os DataFrames da lista empilhando as linhas
df = pd.concat(lista_dfs, ignore_index=True)

print("Tamanho total do dataset unificado:", df.shape)

Encontrados 8 arquivos para importação.
Tamanho total do dataset unificado: (2830743, 79)


In [5]:
# Converte valores infinitos para NaN (nulos)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remove todas as linhas que contenham valores nulos
df.dropna(inplace=True)

print("Tamanho do dataset após remoção de nulos/infinitos:", df.shape)

Tamanho do dataset após remoção de nulos/infinitos: (2827876, 79)


In [6]:
# Separando a variável alvo (Label) dos dados de tráfego
X = df.drop(' Label', axis=1)
y = df[' Label']

# Opcional: Codificação da Label caso seja texto (ex: Benign, DDoS, etc.)
# y = pd.get_dummies(y) ou LabelEncoder

In [7]:
# Aplicação da escala Min-Max conforme exigido para convergência da Rede Neural
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print("Dados numéricos normalizados com sucesso.")

Dados numéricos normalizados com sucesso.


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

print(f"Dados de Treino: {X_train.shape}")
print(f"Dados de Teste: {X_test.shape}")

Dados de Treino: (1979513, 78)
Dados de Teste: (848363, 78)


In [4]:
# Definindo o caminho da nova pasta no Google Drive
caminho_nova_pasta = '/content/drive/MyDrive/Fatec/TCC/dataset filtrado'

nome_dados_treinamento = 'cicids2017_tratado_normalizado_treinamento.csv'
nome_dados_teste = 'cicids2017_tratado_normalizado_teste.csv'

In [9]:
import os

# 1. Reconstruindo o DataFrame com os dados numéricos normalizados
df_tratado = pd.DataFrame(X_scaled, columns=X.columns)
df_test = pd.DataFrame(X_test, columns=X.columns)

# Readicionando a coluna 'Label' ao DataFrame tratado
df_tratado[' Label'] = y.values
df_test[' Label'] = y_test.values

# O comando makedirs cria a pasta "dataset filtrado" caso ela ainda não exista no seu Drive
os.makedirs(caminho_nova_pasta, exist_ok=True)

# 3. Caminho completo do arquivo CSV que será gerado
caminho_arquivo_treinamento = os.path.join(caminho_nova_pasta, nome_dados_treinamento)
caminho_arquivo_teste = os.path.join(caminho_nova_pasta, nome_dados_teste)

# 4. Exportando o DataFrame para o formato CSV
df_tratado.to_csv(caminho_arquivo_treinamento, index=False)
df_test.to_csv(caminho_arquivo_teste, index=False)

print(f"Dataset de treinamento e de teste salvo com sucesso em: {caminho_arquivo_treinamento}")
print(f"Dataset de teste salvo com sucesso em: {caminho_arquivo_teste}")

Dataset de treinamento e de teste salvo com sucesso em: /content/drive/MyDrive/Fatec/TCC/dataset filtrado/cicids2017_tratado_normalizado_treinamento.csv
Dataset de teste salvo com sucesso em: /content/drive/MyDrive/Fatec/TCC/dataset filtrado/cicids2017_tratado_normalizado_teste.csv


In [5]:
# 1. Carregando o input a partir dos CSVs de dados JÁ TRATADOS
print("Carregando os datasets tratados em CSV...")
df_treino = pd.read_csv(caminho_nova_pasta + "/" + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_nova_pasta + "/" + nome_dados_teste)


Carregando os datasets tratados em CSV...


In [6]:
display(df_treino.head())
display(df_teste.head())

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.837186,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,0.840070,1.016667e-06,0.000000,0.000003,4.651163e-07,9.153974e-09,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,0.840085,5.416666e-07,0.000000,0.000003,4.651163e-07,9.153974e-09,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,0.705516,3.916666e-07,0.000000,0.000003,4.651163e-07,9.153974e-09,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,0.837156,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,0.00101,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.585107,9.205005e-02,0.000000,0.000017,4.651163e-07,4.576987e-08,0.000242,0.002581,0.001010,0.000000,...,1.0,0.000342,0.000000,0.000342,0.000342,0.091667,0.000000,0.091667,0.091667,BENIGN
1,0.001221,1.276342e-03,0.000009,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,BENIGN
2,0.006760,5.102475e-01,0.000086,0.000062,1.924031e-04,1.157062e-05,0.024013,0.000000,0.020889,0.026176,...,1.0,0.001497,0.003769,0.006687,0.000459,0.083290,0.000132,0.083333,0.083136,BENIGN
3,0.000809,9.841666e-06,0.000005,0.000007,7.286822e-06,3.356457e-07,0.001894,0.020215,0.007911,0.000000,...,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,BENIGN
4,0.650904,4.666666e-07,0.000000,0.000003,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,BENIGN


In [16]:
# 2. Separando as Features (variáveis numéricas já normalizadas) da Label (gabarito)
X_train = df_treino.drop(' Label', axis=1)
y_train = df_treino[' Label']

X_test = df_teste.drop(' Label', axis=1)
y_test = df_teste[' Label']

print(f"Dados prontos! Treino: {X_train.shape} | Teste: {X_test.shape}")

# 3. Instanciando o modelo Random Forest (conforme metodologia do TCC)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 4. Treinamento medindo o custo computacional
print("\nIniciando o treinamento do Random Forest...")
inicio_treino = time.time()

rf_model.fit(X_train, y_train)

fim_treino = time.time()
tempo_treinamento = fim_treino - inicio_treino
print(f"Treinamento concluído! Tempo total: {tempo_treinamento:.4f} segundos.")

# 5. Classificação (Teste) medindo o tempo de previsão
print("\nIniciando a classificação dos dados de teste...")
inicio_teste = time.time()

previsoes = rf_model.predict(X_test)

fim_teste = time.time()
tempo_teste = fim_teste - inicio_teste
print(f"Classificação concluída! Tempo de previsão: {tempo_teste:.4f} segundos.")

# 6. Avaliação dos Resultados
print("\n=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_test, previsoes))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_test, previsoes))

Dados prontos! Treino: (2827876, 78) | Teste: (848363, 78)

Iniciando o treinamento do Random Forest...
Treinamento concluído! Tempo total: 1686.1174 segundos.

Iniciando a classificação dos dados de teste...
Classificação concluída! Tempo de previsão: 22.0278 segundos.

=== MATRIZ DE CONFUSÃO ===
[[680803      0      1      0     53      0      0      0      0      0
     157      0      0      0      0]
 [     1    563      0      0      0      0      0      0      0      0
       0      0      0      0      0]
 [     0      0  38307      0      0      0      0      0      0      0
       0      0      0      0      0]
 [     0      0      0   3163      0      0      0      0      0      0
       0      0      0      0      0]
 [    14      0      0      0  69253      0      0      0      0      0
       0      0      0      0      0]
 [     0      0      0      0      0   1658      0      0      0      0
       0      0      0      0      0]
 [     0      0      0      0      0     

In [ ]:
# 0. Separando as características (X) do gabarito/rótulo (y)
X_treino = df_treino.drop(' Label', axis=1)
y_treino = df_treino[' Label']

X_teste = df_teste.drop(' Label', axis=1)
y_teste = df_teste[' Label']

# 1. Convertendo as Labels de Texto para Números (Requisito do TensorFlow)
print("Codificando as labels para a Rede Neural...")
encoder = LabelEncoder()
y_train_num = encoder.fit_transform(df_treino)
y_test_num = encoder.transform(df_teste)
num_classes = len(encoder.classes_) # Descobre quantos tipos de ataques existem

# 2. Construindo a Arquitetura da Rede Neural Profunda (DNN)
# Uma arquitetura clássica com duas camadas ocultas
dnn_model = Sequential([
    # Camada Oculta 1 (128 neurônios) conectada à entrada (78 features)
    Dense(128, activation='relu', input_shape=(X_train.shape[4],)),
    Dropout(0.2), # Dropout desliga neurônios aleatórios para evitar "vício" (overfitting)

    # Camada Oculta 2 (64 neurônios)
    Dense(64, activation='relu'),
    Dropout(0.2),

    # Camada de Saída (Um neurônio para cada classe de ataque, usando Softmax para dar a probabilidade)
    Dense(num_classes, activation='softmax')
])

# 3. Compilando o modelo (Definindo como ele vai calcular os erros e se ajustar)
dnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# 4. O Treinamento (Medindo o Custo Computacional)
print("\nIniciando o treinamento da Rede Neural Profunda (DNN)...")
inicio_treino_dnn = time.time()

# O modelo treinará por 10 rodadas completas (epochs), processando os dados em lotes (batch_size)
history = dnn_model.fit(X_train, y_train_num, epochs=10, batch_size=256, validation_split=0.1)

fim_treino_dnn = time.time()
tempo_treinamento_dnn = fim_treino_dnn - inicio_treino_dnn
print(f"\nTreinamento DNN concluído! Tempo total: {tempo_treinamento_dnn:.4f} segundos.")

# 5. Classificação dos Dados de Teste
print("\nIniciando a classificação dos dados de teste...")
inicio_teste_dnn = time.time()

# A rede retorna a probabilidade para cada classe. O argmax pega a classe com a maior probabilidade.
previsoes_probabilidades = dnn_model.predict(X_test)
previsoes_dnn_num = np.argmax(previsoes_probabilidades, axis=1)

fim_teste_dnn = time.time()
tempo_teste_dnn = fim_teste_dnn - inicio_teste_dnn
print(f"Classificação concluída! Tempo de previsão: {tempo_teste_dnn:.4f} segundos.")

# Revertendo os números das previsões de volta para os nomes originais (Benign, DoS, etc.)
previsoes_dnn_labels = encoder.inverse_transform(previsoes_dnn_num)

# 6. Avaliação dos Resultados
print("\n=== MATRIZ DE CONFUSÃO (DNN) ===")
print(confusion_matrix(y_test, previsoes_dnn_labels))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_test, previsoes_dnn_labels))
